# 07 — Robot Portfolio Risk Analysis

Final sinyal ve çıkış kuralları artık sabittir:

- Buy score: 11
- Minimum ADX: 20
- Volume multiplier: 1.30
- Initial stop: 1.5 ATR
- Trailing stop: 2.5 ATR
- Trailing activation: %6

Bu notebook yalnızca portföy seviyesindeki kararları test eder:

- İşlem başına risk
- Maksimum açık pozisyon
- Tek hisse için maksimum portföy payı

Parametre seçimi Development ve Validation dönemleriyle yapılır.
Holdout sonuçları artık görüldüğü için portföy seçimine dahil edilmez.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.config import StrategyConfig, PortfolioConfig
from src.features import add_indicators
from src.signals import build_market_regime
from src.portfolio_experiments import (
    run_portfolio_grid,
    compare_portfolio_periods,
)

sns.set_theme(style="whitegrid")


## 1. Verileri ve final stratejiyi hazırla


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

PERIODS = {
    "Development": ("2018-01-01", "2022-12-31"),
    "Validation": ("2023-01-01", "2024-12-31"),
}

final_strategy = StrategyConfig(
    buy_score=12,
    minimum_adx=18.0,
    volume_multiplier=1.3,
    initial_stop_atr=2.0,
    trailing_stop_atr=2.5,
    trailing_activation_return=0.06,
)

base_portfolio = PortfolioConfig(
    initial_capital=500_000.0,
    commission_rate=0.002,
    slippage_rate=0.002,
)

print("Özellikli hisse verisi:", stock_features.shape)
print(
    "Tarih aralığı:",
    stock_features["Date"].min(),
    "→",
    stock_features["Date"].max(),
)


## 2. Development portföy grid'i

Toplam 27 kombinasyon:

- Risk per trade: %0,30 / %0,50 / %0,75
- Max positions: 6 / 8 / 10
- Tek hisse cap: sınırsız / %15 / %20

`max_position_fraction=None`, mevcut Robot pozisyon büyüklüğü davranışını korur.


In [ ]:
portfolio_grid = {
    "risk_per_trade": [0.003, 0.005, 0.0075],
    "max_positions": [6, 8, 10],
    "max_position_fraction": [None, 0.15, 0.20],
}

portfolio_development = run_portfolio_grid(
    stock_features=stock_features,
    market_regime=market_regime,
    strategy_config=final_strategy,
    base_portfolio=base_portfolio,
    parameter_grid=portfolio_grid,
    start=PERIODS["Development"][0],
    end=PERIODS["Development"][1],
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

portfolio_development.to_csv(
    RESULTS_DIR / "portfolio_grid_development.csv",
    index=False,
)

print("Deney sayısı:", len(portfolio_development))
print(
    "Hata alan deney:",
    portfolio_development["Status"].ne("OK").sum(),
)


## 3. Development dayanıklılık filtresi


In [ ]:
development_candidates = (
    portfolio_development.loc[
        portfolio_development["Status"].eq("OK")
        & portfolio_development["Trade_Count"].ge(150)
        & portfolio_development["Profit_Factor"].ge(1.40)
        & portfolio_development["Max_Drawdown_%"].ge(-40.0)
    ]
    .sort_values(
        [
            "Calmar",
            "Sharpe",
            "Profit_Factor",
            "CAGR_%",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

portfolio_columns = [
    "Experiment_ID",
    "Portfolio_risk_per_trade",
    "Portfolio_max_positions",
    "Portfolio_max_position_fraction",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Calmar",
    "Trade_Count",
    "Exposure_%",
    "Average_Open_Positions",
    "Max_Open_Positions",
    "Average_Invested_%",
]

display(
    development_candidates[
        portfolio_columns
    ].head(15)
)


## 4. Mevcut portföy ayarını da Validation'a taşı

Mevcut ayar:

- Risk: %0,75
- Maksimum pozisyon: 8
- Tek hisse cap: yok


In [ ]:
baseline_mask = (
    portfolio_development[
        "Portfolio_risk_per_trade"
    ].eq(0.0075)
    & portfolio_development[
        "Portfolio_max_positions"
    ].eq(8)
    & portfolio_development[
        "Portfolio_max_position_fraction"
    ].isna()
)

baseline_portfolio_row = portfolio_development.loc[
    baseline_mask
].copy()

selected_portfolios = (
    pd.concat(
        [
            development_candidates.head(10),
            baseline_portfolio_row,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "Portfolio_risk_per_trade",
            "Portfolio_max_positions",
            "Portfolio_max_position_fraction",
        ]
    )
    .reset_index(drop=True)
)

parameter_columns = [
    "Portfolio_risk_per_trade",
    "Portfolio_max_positions",
    "Portfolio_max_position_fraction",
]

portfolio_comparison = compare_portfolio_periods(
    stock_features=stock_features,
    market_regime=market_regime,
    configurations=selected_portfolios,
    strategy_config=final_strategy,
    base_portfolio=base_portfolio,
    periods=PERIODS,
    parameter_columns=parameter_columns,
)

portfolio_comparison.to_csv(
    RESULTS_DIR / "portfolio_selected_validation.csv",
    index=False,
)

comparison_columns = [
    "Selected_Config",
    "Period_Name",
    "Portfolio_risk_per_trade",
    "Portfolio_max_positions",
    "Portfolio_max_position_fraction",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Calmar",
    "Trade_Count",
    "Exposure_%",
    "Average_Open_Positions",
    "Average_Invested_%",
]

display(
    portfolio_comparison[
        comparison_columns
    ].sort_values(
        ["Selected_Config", "Period_Name"]
    )
)


## 5. Development–Validation istikrar tablosu


In [ ]:
portfolio_comparison["Position_Cap_Label"] = (
    portfolio_comparison[
        "Portfolio_max_position_fraction"
    ]
    .map(
        lambda value: (
            "None"
            if pd.isna(value)
            else f"{value:.2f}"
        )
    )
)

portfolio_stability = portfolio_comparison.pivot_table(
    index=[
        "Selected_Config",
        "Portfolio_risk_per_trade",
        "Portfolio_max_positions",
        "Position_Cap_Label",
    ],
    columns="Period_Name",
    values=[
        "CAGR_%",
        "Max_Drawdown_%",
        "Profit_Factor",
        "Sharpe",
        "Calmar",
        "Trade_Count",
        "Exposure_%",
        "Average_Open_Positions",
        "Average_Invested_%",
    ],
    aggfunc="first",
)

portfolio_stability.columns = [
    f"{metric}_{period}"
    for metric, period in portfolio_stability.columns
]

portfolio_stability = portfolio_stability.reset_index()

portfolio_stability["Robust_Calmar"] = portfolio_stability[
    ["Calmar_Development", "Calmar_Validation"]
].min(axis=1)

portfolio_stability["Robust_PF"] = portfolio_stability[
    [
        "Profit_Factor_Development",
        "Profit_Factor_Validation",
    ]
].min(axis=1)

portfolio_stability["Robust_Sharpe"] = portfolio_stability[
    ["Sharpe_Development", "Sharpe_Validation"]
].min(axis=1)

portfolio_stability["Worst_Drawdown"] = portfolio_stability[
    [
        "Max_Drawdown_%_Development",
        "Max_Drawdown_%_Validation",
    ]
].min(axis=1)

portfolio_stability["CAGR_Gap"] = (
    portfolio_stability["CAGR_%_Development"]
    - portfolio_stability["CAGR_%_Validation"]
).abs()

portfolio_stability = portfolio_stability.sort_values(
    [
        "Robust_Calmar",
        "Robust_PF",
        "Robust_Sharpe",
        "CAGR_Gap",
    ],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(portfolio_stability)


## 6. Risk profillerine göre kısa liste

- Conservative: iki dönemde de drawdown en kötü %20
- Balanced: iki dönemde de drawdown en kötü %25
- Aggressive: iki dönemde de drawdown en kötü %32

Her grupta Validation Profit Factor en az 1,40 olmalıdır.


In [ ]:
def risk_bucket(
    df: pd.DataFrame,
    drawdown_floor: float,
) -> pd.DataFrame:
    return (
        df.loc[
            df["Worst_Drawdown"].ge(drawdown_floor)
            & df["Profit_Factor_Validation"].ge(1.40)
            & df["CAGR_%_Validation"].gt(0)
        ]
        .sort_values(
            [
                "Robust_Calmar",
                "Robust_PF",
                "CAGR_Gap",
            ],
            ascending=[False, False, True],
        )
        .head(5)
    )

conservative = risk_bucket(
    portfolio_stability,
    drawdown_floor=-20.0,
)

balanced = risk_bucket(
    portfolio_stability,
    drawdown_floor=-25.0,
)

aggressive = risk_bucket(
    portfolio_stability,
    drawdown_floor=-32.0,
)

print("CONSERVATIVE")
display(conservative)

print("BALANCED")
display(balanced)

print("AGGRESSIVE")
display(aggressive)


## 7. Risk–getiri grafiği


In [ ]:
plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=portfolio_stability,
    x="Worst_Drawdown",
    y="CAGR_%_Validation",
    size="Robust_PF",
    hue="Portfolio_risk_per_trade",
    style="Portfolio_max_positions",
    sizes=(80, 320),
)

plt.axvline(-20, linestyle="--")
plt.axvline(-25, linestyle="--")
plt.axvline(-32, linestyle="--")

plt.title(
    "Portföy Konfigürasyonları — "
    "Validation Getirisi ve En Kötü Drawdown"
)
plt.xlabel(
    "Development/Validation En Kötü Drawdown (%)"
)
plt.ylabel("Validation CAGR (%)")
plt.tight_layout()
plt.show()


## 8. Sonuçları kaydet

Final seçim ilkesi:

1. Önce gerçek hayatta kabul edilebilir drawdown sınırını belirle.
2. O sınır içindeki en yüksek Robust Calmar sonucunu seç.
3. Validation Profit Factor'ın en az 1,40 kalmasını iste.
4. Tek hisse cap kullanılan ve kullanılmayan sonuçları karşılaştır.
5. Seçimden sonra yeni parametre araması yapmadan paper trading'e geç.


In [ ]:
portfolio_stability.to_csv(
    RESULTS_DIR / "portfolio_stability.csv",
    index=False,
)

conservative.to_csv(
    RESULTS_DIR / "portfolio_conservative.csv",
    index=False,
)

balanced.to_csv(
    RESULTS_DIR / "portfolio_balanced.csv",
    index=False,
)

aggressive.to_csv(
    RESULTS_DIR / "portfolio_aggressive.csv",
    index=False,
)

print("Portföy risk analizi sonuçları kaydedildi.")
